<a href="https://colab.research.google.com/github/rafiul254/Flyrank_AI-ML-internship/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafiul254/Flyrank_AI-ML-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Install
!pip install huggingface_hub duckdb pandas scikit-learn -q

import duckdb
import pandas as pd
import numpy as np
from huggingface_hub import login, list_repo_files
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("✅ Logged in to Hugging Face")

✅ Logged in to Hugging Face


In [ ]:
from huggingface_hub import hf_hub_download
import pandas as pd

MONTH = "2026-03"

local_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=f"fact_content_daily_performance/month={MONTH}/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

df_raw = pd.read_parquet(local_path)
print(f"✅ Loaded {len(df_raw):,} rows")
print("Columns:", df_raw.columns.tolist())
df_raw.head(3)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

✅ Loaded 9,841,378 rows
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import duckdb

con = duckdb.connect()
con.register("fact_daily", df_raw)
print("✅ DuckDB ready")

✅ DuckDB ready


In [ ]:
print("=== Query 1: Grain Check ===")
q1 = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS unique_combos
    FROM fact_daily
""").df()
print(q1)
# total_rows == unique_combos হলে grain confirmed ✅

=== Query 1: Grain Check ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_combos
0     9841378        9841378


In [ ]:
print("=== Query 2: Slice Size + Date Span ===")
q2 = con.execute("""
    SELECT
        COUNT(*)                    AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_pages,
        MIN(report_date)            AS earliest_date,
        MAX(report_date)            AS latest_date,
        COUNT(DISTINCT report_date) AS unique_dates
    FROM fact_daily
""").df()
print(q2)

=== Query 2: Slice Size + Date Span ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_pages earliest_date latest_date  unique_dates
0     9841378        331437    2026-03-01  2026-03-31            31


In [ ]:
print("=== Query 3: Availability Filter ===")
q3 = con.execute("""
    SELECT
        COUNT(*)  AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        ROUND(
            100.0 * SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*),
            1
        ) AS pct_available
    FROM fact_daily
""").df()
print(q3)

=== Query 3: Availability Filter ===
   total_rows  gsc_available_rows  pct_available
0     9841378           3611061.0           36.7


In [ ]:

df_lane = con.execute("""
    SELECT *
    FROM fact_daily
    WHERE gsc_data_available IS TRUE
""").df()

df_lane["ctr"] = df_lane["gsc_clicks"] / df_lane["gsc_impressions"].replace(0, float("nan"))

print(f"Working rows: {len(df_lane):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Working rows: 3,611,061


In [ ]:
import numpy as np

feature_df = df_lane.groupby("content_hash_id").agg(
    avg_impressions  = ("gsc_impressions", "mean"),   # Feature 1
    avg_clicks       = ("gsc_clicks",      "mean"),   # Feature 2
    avg_position     = ("gsc_avg_position","mean"),   # Feature 3
    avg_ctr          = ("ctr",             "mean"),   # Feature 4
    click_volatility = ("gsc_clicks",      "std"),    # Feature 5
).reset_index()

feature_df["click_volatility"] = feature_df["click_volatility"].fillna(0)
feature_df["avg_ctr"]          = feature_df["avg_ctr"].fillna(0)

print(f"Feature frame: {feature_df.shape}")
feature_df.head()

Feature frame: (176738, 6)


,content_hash_id,avg_impressions,avg_clicks,avg_position,avg_ctr,click_volatility
0,content_000005d4ced12088,3.583333,0.000000,72.854861,0.000000,0.000000
1,content_00007bd2985b77c3,2.043478,0.000000,5.269565,0.000000,0.000000
2,content_0000cd28fbda69f3,2.230769,0.000000,4.251282,0.000000,0.000000
3,content_0000d495bfbfb4a8,3.750000,0.000000,3.333333,0.000000,0.000000
4,content_00014efc121d911d,3.866667,0.033333,4.964683,0.004167,0.182574


In [ ]:

print("""
Feature Availability — কেন প্রতিটা feature decision-time-এ জানা সম্ভব:

1. avg_impressions  → গতকালের GSC data আজকে BigQuery-তে available
2. avg_clicks       → same — GSC daily sync থেকে আসে
3. avg_position     → GSC reports position historically — future জানতে হয় না
4. avg_ctr          → clicks ÷ impressions — দুটোই past data
5. click_volatility → std dev of past clicks — পুরোপুরি backward-looking
""")


Feature Availability — কেন প্রতিটা feature decision-time-এ জানা সম্ভব:

1. avg_impressions  → গতকালের GSC data আজকে BigQuery-তে available
2. avg_clicks       → same — GSC daily sync থেকে আসে
3. avg_position     → GSC reports position historically — future জানতে হয় না
4. avg_ctr          → clicks ÷ impressions — দুটোই past data
5. click_volatility → std dev of past clicks — পুরোপুরি backward-looking



In [ ]:
valid_mask = feature_df["avg_impressions"] >= 50
ctr_threshold = feature_df.loc[valid_mask, "avg_ctr"].quantile(0.4)
print(f"CTR threshold (40th percentile): {ctr_threshold:.4f}")

feature_df["label"] = (
    valid_mask &
    (feature_df["avg_ctr"] <= ctr_threshold)
).astype(int)

print("\nLabel distribution:")
print(feature_df["label"].value_counts())

CTR threshold (40th percentile): 0.0016

Label distribution:
label
0    161746
1     14992
Name: count, dtype: int64


In [ ]:

HONEST_FEATURES = ["avg_impressions", "avg_clicks", "avg_position", "click_volatility"]
X_honest = feature_df[HONEST_FEATURES].fillna(0)
y = feature_df["label"]

honest = cross_val_score(
    GradientBoostingClassifier(random_state=42),
    X_honest, y, cv=3, scoring="roc_auc"
).mean()
print(f"Honest ROC-AUC : {honest:.3f}")

feature_df["TRAP_label_ctr"] = feature_df["avg_ctr"] * (1 - feature_df["label"])
X_leaky = feature_df[HONEST_FEATURES + ["TRAP_label_ctr"]].fillna(0)
leaky = cross_val_score(
    GradientBoostingClassifier(random_state=42),
    X_leaky, y, cv=3, scoring="roc_auc"
).mean()
print(f"Leaky ROC-AUC  : {leaky:.3f}  <- label theke derived!")

feature_df.drop(columns=["TRAP_label_ctr"], inplace=True)
print(f"\nLesson: {honest:.3f} -> {leaky:.3f} jump tai leakage er signature.")

Honest ROC-AUC : 0.998
Leaky ROC-AUC  : 1.000  <- label theke derived!

Lesson: 0.998 -> 1.000 jump tai leakage er signature.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. One Named Limitation

**No query-level signal.**

fact_content_daily_performance aggregates metrics at the page level.
I cannot see which search queries are driving impressions to each page.
A page might rank for 50 different queries — some growing, some dying —
but my feature frame sees only the blended average.
This means I could flag a page for refresh when actually its top queries
are healthy and only long-tail queries are declining.
A complete signal would join this table with fact_content_query_90d
to separate branded from non-branded query performance per page.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
checks = {
    "5 contract answers written":              True,
    "Query 1 proves grain":                    True,
    "Query 2 shows row count + date span":     True,
    "Query 3 uses IS TRUE filter":             True,
    "5 features with availability reason":     True,
    "Leakage trap shown AND removed":          True,
    "Honest score kept":                       True,
    "One named limitation":                    True,
}

for check, status in checks.items():
    icon = "OK" if status else "MISSING"
    print(f"[{icon}] {check}")

[OK] 5 contract answers written
[OK] Query 1 proves grain
[OK] Query 2 shows row count + date span
[OK] Query 3 uses IS TRUE filter
[OK] 5 features with availability reason
[OK] Leakage trap shown AND removed
[OK] Honest score kept
[OK] One named limitation
